# Sección 5 – LightGBM Global con Rezagos y Calendario

Este notebook implementa un **modelo global LightGBM** que aprende de todos los clientes simultáneamente.

**Características del modelo:**
- Rezagos: `lag_1` … `lag_24` (un día completo), más `lag_168` (mismo instante la semana anterior)
- Covariables de calendario: seno/coseno de la hora, día de la semana, mes, indicador de feriado
- Identificador de cliente (`unique_id`) codificado numéricamente
- Entrenamiento global: un único modelo para los N clientes

**Protocolo:** backtesting con origen móvil diario usando `backtest()` de `src/backtest.py`.

## 0. Configuración e importaciones

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Raíz del proyecto
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import lightgbm as lgb

from src.backtest import backtest, ForecastModel
from src.metrics import evaluate_all, mae_per_horizon

# ── Rutas ──────────────────────────────────────────────────────────────────
DATA_PATH   = os.path.join(PROJECT_ROOT, 'data', 'processed', 'panel_hourly.parquet')
OUTPUT_DIR  = os.path.join(PROJECT_ROOT, 'outputs')
FIGURES_DIR = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Parámetros globales ────────────────────────────────────────────────────
TRAIN_END  = '2014-09-30'
TEST_START = '2014-10-01'
TEST_END   = '2014-12-31'
H          = 24   # horizonte (horas)

# Semilla para reproducibilidad
SEED = 42

print('Project root:', PROJECT_ROOT)
print('LightGBM version:', lgb.__version__)

## 1. Carga del panel

In [ ]:
panel = pd.read_parquet(DATA_PATH)
panel.index = pd.to_datetime(panel.index)
panel = panel.asfreq('h')

print(f'Panel shape  : {panel.shape}  (T={panel.shape[0]} horas, N={panel.shape[1]} clientes)')
print(f'Rango temporal: {panel.index[0]}  →  {panel.index[-1]}')

# Separar train / test
panel_train = panel.loc[:TRAIN_END]
y_train     = panel_train.values.T   # (N, T_train)

# Mapa cliente → índice entero (para LightGBM)
clients       = list(panel.columns)
client_to_idx = {c: i for i, c in enumerate(clients)}
N_CLIENTS     = len(clients)

print(f'Clientes     : {N_CLIENTS}')

## 2. Feature Engineering

Construimos todas las características a partir de un panel (T x N):

| Grupo          | Características                                                |
|----------------|----------------------------------------------------------------|
| Rezagos        | `lag_1` … `lag_24`, `lag_168`                                  |
| Hora del día   | `sin_hour`, `cos_hour` (codificación cíclica)                  |
| Calendario     | `day_of_week`, `month`, `is_weekend`, `is_holiday`             |
| Identidad      | `client_id` (entero)                                           |

In [ ]:
# ── Conjunto de feriados (Irlanda / CER dataset) ───────────────────────────
# Ajustar esta lista según el país/dataset real
IRISH_HOLIDAYS = pd.to_datetime([
    # 2013
    '2013-01-01', '2013-03-17', '2013-03-29', '2013-04-01',
    '2013-05-06', '2013-06-03', '2013-08-05', '2013-10-28',
    '2013-12-25', '2013-12-26',
    # 2014
    '2014-01-01', '2014-03-17', '2014-04-18', '2014-04-21',
    '2014-05-05', '2014-06-02', '2014-08-04', '2014-10-27',
    '2014-12-25', '2014-12-26',
])

HOLIDAY_SET = set(IRISH_HOLIDAYS.normalize())  # solo fecha, sin hora


def build_features(
    panel: pd.DataFrame,
    client_to_idx: dict,
    lags: list = None,
    target_col: str = 'y',
) -> pd.DataFrame:
    """
    Construye la tabla de features en formato largo a partir del panel (T x N).

    Args:
        panel        : DataFrame (T x N) con frecuencia horaria.
        client_to_idx: mapeo cliente → entero.
        lags         : lista de rezagos a calcular. Por defecto lag_1..24 + lag_168.
        target_col   : nombre de la columna objetivo.

    Returns:
        DataFrame con columnas [client_id, ds, lag_*, sin_hour, cos_hour,
                                day_of_week, month, is_weekend, is_holiday, y]
    """
    if lags is None:
        lags = list(range(1, 25)) + [168]   # lag_1..24 + lag semanal

    rows = []

    for client in panel.columns:
        series = panel[client].copy()
        df_c   = pd.DataFrame({'ds': series.index, 'y': series.values})

        # ── Rezagos ────────────────────────────────────────────────────────
        for lag in lags:
            df_c[f'lag_{lag}'] = df_c['y'].shift(lag)

        # ── Covariables de calendario ──────────────────────────────────────
        df_c['hour']        = df_c['ds'].dt.hour
        df_c['sin_hour']    = np.sin(2 * np.pi * df_c['hour'] / 24)
        df_c['cos_hour']    = np.cos(2 * np.pi * df_c['hour'] / 24)
        df_c['sin_dow']     = np.sin(2 * np.pi * df_c['ds'].dt.dayofweek / 7)
        df_c['cos_dow']     = np.cos(2 * np.pi * df_c['ds'].dt.dayofweek / 7)
        df_c['day_of_week'] = df_c['ds'].dt.dayofweek    # 0=Lunes
        df_c['month']       = df_c['ds'].dt.month
        df_c['is_weekend']  = (df_c['day_of_week'] >= 5).astype(int)
        df_c['is_holiday']  = df_c['ds'].dt.normalize().isin(HOLIDAY_SET).astype(int)
        df_c['client_id']   = client_to_idx[client]

        rows.append(df_c)

    features = pd.concat(rows, ignore_index=True)
    features = features.dropna()   # eliminar filas sin rezagos completos
    return features


# ── Construcción de features de entrenamiento ──────────────────────────────
print('Construyendo features de entrenamiento…')
LAGS = list(range(1, 25)) + [168]
features_train = build_features(panel_train, client_to_idx, lags=LAGS)

print(f'Shape features train: {features_train.shape}')
features_train.head(3)

In [ ]:
# ── Definir columnas de features y target ─────────────────────────────────
FEATURE_COLS = (
    [f'lag_{l}' for l in LAGS]
    + ['sin_hour', 'cos_hour', 'sin_dow', 'cos_dow',
       'day_of_week', 'month', 'is_weekend', 'is_holiday',
       'client_id']
)
TARGET_COL = 'y'

print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

## 3. Entrenamiento del modelo global LightGBM

In [ ]:
# ── Hiperparámetros ────────────────────────────────────────────────────────
# ↕ Ajustar estos valores para mejorar el rendimiento
LGB_PARAMS = {
    'objective'        : 'regression_l1',  # MAE como función objetivo
    'metric'           : 'mae',
    'n_estimators'     : 1000,             # ↕ aumentar para datasets grandes
    'learning_rate'    : 0.05,             # ↕ reducir al aumentar n_estimators
    'num_leaves'       : 127,              # ↕ complejidad del árbol
    'max_depth'        : -1,               # sin límite
    'min_child_samples': 20,               # ↕ regularización mínima por hoja
    'subsample'        : 0.8,              # ↕ fracción de filas por árbol
    'colsample_bytree' : 0.8,             # ↕ fracción de columnas por árbol
    'reg_alpha'        : 0.1,             # ↕ regularización L1
    'reg_lambda'       : 0.1,             # ↕ regularización L2
    'random_state'     : SEED,
    'n_jobs'           : -1,
    'verbose'          : -1,
}

X_train = features_train[FEATURE_COLS].values
y_trn   = features_train[TARGET_COL].values

# ── Entrenar con early stopping usando 10% como validación interna ─────────
val_size = int(len(X_train) * 0.1)
X_val, y_val = X_train[-val_size:], y_trn[-val_size:]
X_fit, y_fit = X_train[:-val_size],  y_trn[:-val_size]

print(f'Train samples : {len(X_fit):,}')
print(f'Val   samples : {len(X_val):,}')

lgb_model = lgb.LGBMRegressor(**LGB_PARAMS)
lgb_model.fit(
    X_fit, y_fit,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=100),
    ],
)
print(f'\nMejor iteración: {lgb_model.best_iteration_}')

## 4. Wrapper `LightGBMGlobalModel` (protocolo `ForecastModel`)

In [ ]:
class LightGBMGlobalModel:
    """
    Wrapper del modelo global LightGBM para el protocolo ForecastModel.

    Estrategia de pronóstico recursivo:
        Para h=1..H, se predice un paso y se añade la predicción
        al historial para calcular los rezagos del siguiente paso.
    """

    def __init__(
        self,
        model: lgb.LGBMRegressor,
        client_to_idx: dict,
        feature_cols: list,
        lags: list,
        holiday_set: set,
        model_name: str = 'LightGBM_Global',
    ):
        self._model        = model
        self._client_idx   = client_to_idx
        self._feature_cols = feature_cols
        self._lags         = lags
        self._holiday_set  = holiday_set
        self._name         = model_name
        self._max_lag      = max(lags)

    @property
    def name(self) -> str:
        return self._name

    def _make_calendar_features(self, ts: pd.Timestamp) -> dict:
        """Retorna dict con las covariables de calendario para el instante ts."""
        h = ts.hour
        return {
            'sin_hour'   : np.sin(2 * np.pi * h / 24),
            'cos_hour'   : np.cos(2 * np.pi * h / 24),
            'sin_dow'    : np.sin(2 * np.pi * ts.dayofweek / 7),
            'cos_dow'    : np.cos(2 * np.pi * ts.dayofweek / 7),
            'day_of_week': ts.dayofweek,
            'month'      : ts.month,
            'is_weekend' : int(ts.dayofweek >= 5),
            'is_holiday' : int(ts.normalize() in self._holiday_set),
        }

    def predict(
        self,
        history: pd.DataFrame,
        horizon: int,
        origin_time: pd.Timestamp,
    ) -> np.ndarray:
        """
        Genera pronósticos recursivos paso a paso para todos los clientes.

        Args:
            history     : DataFrame (T x N) con datos hasta el cutoff.
            horizon     : número de pasos a pronosticar.
            origin_time : timestamp del primer paso (00:00 del día de prueba).

        Returns:
            array (N, H) con predicciones en kWh, clipadas a 0.
        """
        clients  = list(history.columns)
        n        = len(clients)
        preds    = np.zeros((n, horizon))

        # Buffer de historial para actualizarlo recursivamente
        # Guardamos solo los últimos max_lag valores por cliente
        history_buffer = {
            c: list(history[c].dropna().values[-self._max_lag:])
            for c in clients
        }

        for h_step in range(horizon):
            ts = origin_time + pd.Timedelta(hours=h_step)
            cal = self._make_calendar_features(ts)

            # Construir fila de features para cada cliente
            X_rows = []
            for c_idx, client in enumerate(clients):
                buf = history_buffer[client]
                lag_feats = {}
                for lag in self._lags:
                    idx = len(buf) - lag
                    lag_feats[f'lag_{lag}'] = buf[idx] if idx >= 0 else 0.0

                row = {**lag_feats, **cal, 'client_id': self._client_idx[client]}
                X_rows.append([row[f] for f in self._feature_cols])

            X_step = np.array(X_rows, dtype=np.float32)  # (N, n_features)

            # Predecir un paso para todos los clientes a la vez
            y_step = self._model.predict(X_step)          # (N,)
            y_step = np.clip(y_step, 0, None)

            preds[:, h_step] = y_step

            # Actualizar buffer con la predicción (pronóstico recursivo)
            for c_idx, client in enumerate(clients):
                history_buffer[client].append(float(y_step[c_idx]))

        return preds


# Instanciar el wrapper
lgbm_wrapper = LightGBMGlobalModel(
    model=lgb_model,
    client_to_idx=client_to_idx,
    feature_cols=FEATURE_COLS,
    lags=LAGS,
    holiday_set=HOLIDAY_SET,
    model_name='LightGBM_Global',
)
print(f'Modelo creado: {lgbm_wrapper.name}')

## 5. Backtesting

In [ ]:
print('Ejecutando backtest…')
result = backtest(
    model=lgbm_wrapper,
    panel=panel,
    y_train=y_train,
    H=H,
    test_start=TEST_START,
    test_end=TEST_END,
    seasonality=24,
    return_predictions=True,
    verbose=True,
)

metrics   = result['metrics']
mae_h     = result['mae_horizon']
preds_df  = result['predictions']

print(f'\nMAE  : {metrics["MAE"]:.4f} kWh')
print(f'RMSE : {metrics["RMSE"]:.4f} kWh')
print(f'WAPE : {metrics["WAPE"]:.4f}')
print(f'MASE : {metrics["MASE"]:.4f}')
print(f'Tiempo de inferencia: {result["inference_time"]:.1f}s')

## 6. Guardar métricas y predicciones

In [ ]:
# ── 6.1 Métricas ──────────────────────────────────────────────────────────
metrics_df = pd.DataFrame([metrics])
if 'model' not in metrics_df.columns:
    metrics_df.insert(0, 'model', lgbm_wrapper.name)

# Intentar cargar métricas de baselines para comparación
baseline_metrics_path = os.path.join(OUTPUT_DIR, 'metrics_baselines.csv')
if os.path.exists(baseline_metrics_path):
    baseline_df = pd.read_csv(baseline_metrics_path)
    combined_df = pd.concat([baseline_df, metrics_df], ignore_index=True)
    print('Métricas combinadas (baselines + LightGBM):')
    print(combined_df.to_string(index=False))
else:
    print('Métricas LightGBM:')
    print(metrics_df.to_string(index=False))

metrics_path = os.path.join(OUTPUT_DIR, 'metrics_lightgbm.csv')
metrics_df.to_csv(metrics_path, index=False)
print(f'\nMétricas guardadas en: {metrics_path}')

In [ ]:
# ── 6.2 Predicciones ──────────────────────────────────────────────────────
# Asegurar columnas: (unique_id, ds, modelo, y_hat)
preds_df = preds_df[['unique_id', 'ds', 'modelo', 'y_hat']]

preds_path = os.path.join(OUTPUT_DIR, 'predictions_lightgbm.csv')
preds_df.to_csv(preds_path, index=False)
print(f'Predicciones guardadas en: {preds_path}')
print(f'Filas totales: {len(preds_df):,}')
preds_df.head()

## 7. Importancia de Features

In [ ]:
# ── 7.1 Tabla de importancia ───────────────────────────────────────────────
importance_df = pd.DataFrame({
    'feature'   : FEATURE_COLS,
    'importance': lgb_model.feature_importances_,
}).sort_values('importance', ascending=False)

print('Top 15 features:')
importance_df.head(15).to_string(index=False)

In [ ]:
# ── 7.2 Gráfico de importancia ─────────────────────────────────────────────
TOP_N = 20   # número de features a mostrar

top_imp = importance_df.head(TOP_N).sort_values('importance')

# Categorizar features por tipo para el color
def feature_color(feat):
    if feat.startswith('lag_'):
        lag_val = int(feat.split('_')[1])
        return '#e74c3c' if lag_val == 168 else '#3498db'
    elif feat in ('sin_hour', 'cos_hour'):
        return '#2ecc71'
    elif feat in ('sin_dow', 'cos_dow', 'day_of_week', 'is_weekend'):
        return '#9b59b6'
    elif feat == 'client_id':
        return '#f39c12'
    else:
        return '#95a5a6'

colors = [feature_color(f) for f in top_imp['feature']]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top_imp['feature'], top_imp['importance'], color=colors, edgecolor='white')

ax.set_xlabel('Importancia (splits)', fontsize=12)
ax.set_title(f'Top {TOP_N} Features – LightGBM Global', fontsize=14, fontweight='bold')
ax.grid(True, axis='x', linestyle='--', alpha=0.5)

# Leyenda de colores
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', label='Rezagos cortos (lag_1..24)'),
    Patch(facecolor='#e74c3c', label='Rezago semanal (lag_168)'),
    Patch(facecolor='#2ecc71', label='Hora (sin/cos)'),
    Patch(facecolor='#9b59b6', label='Día semana / fin de semana'),
    Patch(facecolor='#f39c12', label='Identidad del cliente'),
    Patch(facecolor='#95a5a6', label='Otros (mes, feriado…)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, framealpha=0.9)

fig.tight_layout()
fig_path = os.path.join(FIGURES_DIR, 'feature_importance_lightgbm.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Figura guardada en: {fig_path}')
plt.show()

## 8. MAE por paso del horizonte

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    range(1, H + 1), mae_h,
    color='#e74c3c', marker='o', markersize=5, linewidth=2, label='LightGBM_Global',
)

ax.set_xlabel('Paso del horizonte (h)', fontsize=12)
ax.set_ylabel('MAE (kWh)', fontsize=12)
ax.set_title('MAE por paso del horizonte — LightGBM Global', fontsize=14, fontweight='bold')
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()

fig.tight_layout()
fig_mae_path = os.path.join(FIGURES_DIR, 'mae_horizon_lightgbm.png')
fig.savefig(fig_mae_path, dpi=150, bbox_inches='tight')
print(f'Figura guardada en: {fig_mae_path}')
plt.show()

## 9. Análisis de predicciones — muestra de clientes

In [ ]:
# Visualizar pronóstico vs. real para 3 clientes en una semana de prueba
sample_clients = clients[:3]
SAMPLE_WEEK_START = '2014-10-06'
SAMPLE_WEEK_END   = '2014-10-12'

preds_df['ds'] = pd.to_datetime(preds_df['ds'])
mask_week = (preds_df['ds'] >= SAMPLE_WEEK_START) & (preds_df['ds'] <= SAMPLE_WEEK_END)

fig, axes = plt.subplots(len(sample_clients), 1, figsize=(14, 4 * len(sample_clients)), sharex=True)

for ax, client in zip(axes, sample_clients):
    # Real
    real = panel.loc[SAMPLE_WEEK_START:SAMPLE_WEEK_END, client]
    ax.plot(real.index, real.values, label='Real', color='#2c3e50', linewidth=1.5)

    # Predicción
    pred = preds_df[(preds_df['unique_id'] == client) & mask_week]
    ax.plot(pred['ds'], pred['y_hat'], label='LightGBM', color='#e74c3c',
            linewidth=1.5, linestyle='--')

    ax.set_title(f'Cliente: {client}', fontsize=11)
    ax.set_ylabel('kWh', fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Fecha', fontsize=11)
fig.suptitle(f'Pronóstico vs. Real — Semana del {SAMPLE_WEEK_START}', fontsize=13, fontweight='bold')
fig.tight_layout()

fig_sample_path = os.path.join(FIGURES_DIR, 'sample_predictions_lightgbm.png')
fig.savefig(fig_sample_path, dpi=150, bbox_inches='tight')
print(f'Figura guardada en: {fig_sample_path}')
plt.show()

## 10. Resumen final

In [ ]:
print('='*65)
print('RESUMEN – LightGBM Global')
print('='*65)
print(f'MAE  : {metrics["MAE"]:.4f} kWh')
print(f'RMSE : {metrics["RMSE"]:.4f} kWh')
print(f'WAPE : {metrics["WAPE"]:.4f}')
print(f'MASE : {metrics["MASE"]:.4f}')
print(f'Tiempo inferencia: {result["inference_time"]:.1f}s')
print()
print('Top 5 features por importancia:')
for _, row in importance_df.head(5).iterrows():
    print(f'  {row["feature"]:25s}  {row["importance"]:8.0f}')
print()
print('Archivos generados:')
print(f'  → {metrics_path}')
print(f'  → {preds_path}')
print(f'  → {fig_path}')
print(f'  → {fig_mae_path}')
print(f'  → {fig_sample_path}')